# Field-Field PPC — v5: AR whitening + per-rat pooling + paired cluster permutation

Identical to v4's PPC section, with one preprocessing change: each HPC and PFC segment is AR-whitened before the wavelet / cross-spectrum step. The whitening is a Python port of `bz_whitenLFP.m` (buzcode, AntonioFR 5/20) — AR(2) by default, common AR model across channels, mirror-padded zero-lag filtering (`filt0`).

Rationale: raw LFPs have a ~1/f broadband backbone that dominates the cross-spectrum and can smear / bias phase-binned coupling estimates toward low frequencies. Whitening flattens that backbone so that oscillatory coupling at higher frequencies is not masked. Theta cycle detection stays on the original EMD theta IMF — whitening a narrow band distorts it — so only the broadband signals that go into the wavelet are whitened.

Pickle paths use a `_whiten` suffix so the v4 outputs are not overwritten.

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import os, sys, re, copy, pickle
import pandas as pd
import seaborn as sns
from tqdm import tqdm

import scipy
import scipy.stats
import scipy.signal
import scipy.linalg
from scipy.signal import hilbert
from scipy.io import loadmat

import sails
import emd
import emd.sift as sift
from neurodsp.filt import filter_signal

import mne
from mne.stats import permutation_cluster_1samp_test

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

BASE_PATH = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition'
FS = 1000
FREQUENCIES = np.arange(15, 181, 1)
N_PHASE_BINS = 19
N_CYCLES_WAVELET = 5

AR_ORDER = 2  # whitening order; bz_whitenLFP default

## Helpers (lifted verbatim from v2/v4)

In [ ]:
def extract_lfp_by_pt_intervals(lfp, fs, interval):
    out = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        out.append(np.array(lfp[start_idx:end_idx]))
    return out


def find_file_local(directory, prefix):
    for fname in os.listdir(directory):
        low = fname.lower()
        if low.startswith(prefix.lower()) and low.endswith('.mat'):
            return os.path.join(directory, fname)
        if prefix == 'states' and 'states' in low and low.endswith('.mat'):
            return os.path.join(directory, fname)
    return None

## LFP whitening — Python port of `bz_whitenLFP.m`

```matlab
[k, Atmp] = arfit(x, p, p);
A = [1 -Atmp];
Lout = filt0(x, A);
```

We fit an AR(p) model to the signal (default p=2) and apply the inverse filter `W = [1, -a_1, ..., -a_p]` to get the whitened residual. `filt0` is MATLAB's zero-lag FIR with mirror-padded edges.

Implementation notes:
- `arfit` (stepwise LS, Schneider & Neumaier) is not in core scipy. We use **Yule–Walker** with biased autocorrelations, solved via `scipy.linalg.solve_toeplitz`. For p=2 on long LFP segments the two estimators agree to within numerical noise.
- `commonAR=True` (default): fit the AR model on the first channel of the first segment and reuse for all channels. `commonAR=False`: fit per channel.
- `window=None` (default): fit one AR model over the whole segment. If `window` is given, one model per window.
- If `ar_model` is passed, skip fitting and use it directly (useful to reuse a model across HPC/PFC or across phasic/tonic).

In [ ]:
def _fit_ar_yule_walker(x, order):
    """Fit AR(order) model by Yule-Walker. Returns `rho` s.t. x[t] ~ sum_k rho[k-1]*x[t-k].
    Mirrors what `arfit(x,p,p)` returns (the Atmp vector)."""
    x = np.asarray(x, dtype=float).ravel()
    x = x - x.mean()
    n = x.size
    # biased autocorrelation, lags 0..order (same convention as arfit / arburg on long signals)
    r = np.empty(order + 1)
    for k in range(order + 1):
        r[k] = np.dot(x[: n - k], x[k:]) / n
    if r[0] <= 0 or not np.isfinite(r[0]):
        return np.zeros(order)
    rho = scipy.linalg.solve_toeplitz(r[:order], r[1:order + 1])
    return rho


def _filt0(x, W):
    """Zero-lag FIR filter with mirror-padded edges. Port of MATLAB filt0 in bz_whitenLFP."""
    W = np.asarray(W, dtype=float).ravel()
    if W.size < 2:
        raise ValueError('W must be a vector of length >= 2')
    x = np.asarray(x, dtype=float)
    squeeze_back = False
    if x.ndim == 1:
        x = x[:, None]
        squeeze_back = True
    n_samples, n_ch = x.shape
    C = W.size
    if C > n_samples:
        out = np.full_like(x, np.nan)
        return out.ravel() if squeeze_back else out
    D = int(np.ceil(C / 2)) - 1
    head = np.flipud(x[:C, :])
    tail = np.flipud(x[-C:, :])
    x_pad = np.concatenate([head, x, tail], axis=0)
    Y = scipy.signal.lfilter(W, 1.0, x_pad, axis=0)
    start = C + D
    end = start + n_samples
    Y = Y[start:end, :]
    return Y.ravel() if squeeze_back else Y


def whiten_lfp(lfp, fs=None, ar_order=2, window=None, common_ar=True, ar_model=None):
    """AR-whiten an LFP signal. Python port of bz_whitenLFP.m (AntonioFR, 5/20).

    Parameters
    ----------
    lfp : 1D or 2D array (n_samples,) or (n_samples, n_channels), or dict with
          keys {'data','timestamps','samplingRate'}.
    fs : sampling rate (only needed when `lfp` is a plain array and you care
         about the returned timestamps).
    ar_order : int. Default 2.
    window : int samples. If given, refits AR in windows of that size.
             If None, one model for the whole segment.
    common_ar : if True, fit model on channel 0 of the first window and reuse.
    ar_model : precomputed filter vector W=[1,-a_1,...,-a_p]. If given, skip fitting.

    Returns
    -------
    out : same shape as input (1D or 2D). Whitened signal.
    A   : the filter vector W used (for reuse across segments).
    """
    if isinstance(lfp, dict):
        data = np.asarray(lfp['data'], dtype=float)
    else:
        data = np.asarray(lfp, dtype=float)

    squeeze_back = False
    if data.ndim == 1:
        data = data[:, None]
        squeeze_back = True

    n_samples, n_ch = data.shape
    out = np.zeros_like(data)

    if window is None:
        segs = [(0, n_samples)]
    else:
        window = int(window)
        nwin = n_samples // window + 1
        segs = []
        for i in range(nwin):
            s = i * window
            e = min(s + window, n_samples)
            if s < e:
                segs.append((s, e))

    A = np.asarray(ar_model, dtype=float).ravel() if ar_model is not None else None

    for (s, e) in segs:
        if A is not None:
            for i in range(n_ch):
                out[s:e, i] = _filt0(data[s:e, i], A)
        elif common_ar:
            if A is None:
                rho = _fit_ar_yule_walker(data[s:e, 0], ar_order)
                A = np.concatenate([[1.0], -rho])
            for i in range(n_ch):
                out[s:e, i] = _filt0(data[s:e, i], A)
        else:
            for i in range(n_ch):
                rho = _fit_ar_yule_walker(data[s:e, i], ar_order)
                A_i = np.concatenate([[1.0], -rho])
                out[s:e, i] = _filt0(data[s:e, i], A_i)

    if squeeze_back:
        out = out.ravel()
    return out, A

### Quick sanity check — AR(2) removes 1/f slope

In [ ]:
# synthesize a long AR(2) process with a superimposed 8 Hz oscillation and
# verify: (a) the whitening filter matches the generating coefficients,
# (b) the whitened PSD is roughly flat apart from the injected peak.
rng = np.random.default_rng(0)
T, fs = 30_000, 1000
a1, a2 = 1.2, -0.6  # gives an AR(2) with 1/f-ish low-freq backbone
x = np.zeros(T)
innov = rng.standard_normal(T)
for t in range(2, T):
    x[t] = a1 * x[t-1] + a2 * x[t-2] + innov[t]
x += 0.5 * np.sin(2 * np.pi * 8 * np.arange(T) / fs)

x_w, A = whiten_lfp(x, fs=fs, ar_order=2)
print('fitted filter W =', A, '  (expected ~ [1,', -a1, ',', -a2, '])')

from scipy.signal import welch
f, P0 = welch(x, fs=fs, nperseg=2048)
_, P1 = welch(x_w, fs=fs, nperseg=2048)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(f[1:], P0[1:], label='raw')
ax.loglog(f[1:], P1[1:], label='AR(2)-whitened')
ax.set_xlabel('Hz'); ax.set_ylabel('PSD'); ax.legend(); plt.tight_layout(); plt.show()

## Core PPC — with AR whitening on HPC and PFC before wavelet

Only the broadband signals that go into the wavelet are whitened. The theta IMF that drives cycle detection is the untouched EMD output.

In [ ]:
def compute_field_field_ppc(hpc_imfs, pfc_lfp_segments, fs=1000,
                            frequencies=np.arange(15, 141, 1),
                            n_phase_bins=20, n_cycles_wavelet=5,
                            ar_order=AR_ORDER, whiten=True,
                            verbose=True):
    """
    Field-field PPC(f, phi) between HPC and PFC, Zhang et al. (2019).
    v5: each HPC and PFC segment is AR-whitened (order=`ar_order`) before the
    wavelet transform. Theta IMF / cycle detection is unchanged.
    """
    all_W = []

    iterator = range(len(hpc_imfs))
    if verbose:
        iterator = tqdm(iterator, desc='PPC intervals', leave=False)

    for idx in iterator:
        imf = hpc_imfs[idx]
        pfc = pfc_lfp_segments[idx]

        hpc = np.sum(imf, axis=1)
        min_len = min(len(hpc), len(pfc))
        hpc, pfc = hpc[:min_len], pfc[:min_len]
        theta_imf = imf[:min_len, 5]

        cycle_data = get_cycle_data(theta_imf, fs)
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        conditions = [
            'is_good==1',
            f'duration_samples<{fs / 5}',
            f'duration_samples>{fs / 12}',
            f'max_amp>{amp_thresh}'
        ]
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            continue

        subset_df = all_cycles.get_metric_dataframe(subset=True)
        cycle_inds = arrange_cycle_inds(
            get_cycle_inds(all_cycles, subset_df['index'].values)
        )

        if whiten:
            try:
                hpc_in, _ = whiten_lfp(hpc, fs=fs, ar_order=ar_order)
                pfc_in, _ = whiten_lfp(pfc, fs=fs, ar_order=ar_order)
            except Exception as e:
                if verbose:
                    print(f'  [warn] whitening failed ({e}); using raw signal')
                hpc_in, pfc_in = hpc, pfc
        else:
            hpc_in, pfc_in = hpc, pfc

        hpc_cwt = sails.wavelet.morlet(
            hpc_in, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )
        pfc_cwt = sails.wavelet.morlet(
            pfc_in, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )
        cross = hpc_cwt * np.conj(pfc_cwt)

        for c in range(len(cycle_inds)):
            start, end = cycle_inds[c]
            cycle_len = end - start
            if end > min_len or cycle_len < n_phase_bins:
                continue
            cs_cycle = cross[:, start:end]
            bin_edges = np.linspace(0, cycle_len, n_phase_bins + 1, dtype=int)
            B_k = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
            for pb in range(n_phase_bins):
                b_s, b_e = bin_edges[pb], bin_edges[pb + 1]
                if b_e > b_s:
                    B_k[:, pb] = np.mean(cs_cycle[:, b_s:b_e], axis=1)
            W_k = np.angle(B_k)
            all_W.append(W_k)

    N = len(all_W)
    if verbose:
        print(f'  total theta cycles: {N}')
    if N < 2:
        return None, frequencies, N

    sum_vec = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
    for W_k in all_W:
        sum_vec += np.exp(1j * W_k)

    ppc = (np.abs(sum_vec) ** 2 - N) / (N * (N - 1))
    return ppc, frequencies, N

## `compute_ppc_per_rat` — pool cycles within rat × condition

Same driver as v4 (and calls the whitened PPC above).

In [ ]:
def _collect_rat_intervals(rat_id, base_path, condition_filter=None, fs=1000, cfg=None):
    preferred = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    if condition_filter is not None:
        conditions = [c for c in preferred if c in condition_filter and os.path.isdir(os.path.join(base_path, c))]
    else:
        conditions = [c for c in preferred if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    phasic_imfs_all, phasic_pfc_all = [], []
    tonic_imfs_all, tonic_pfc_all = [], []
    session_log = []

    for cond in conditions:
        cond_path = os.path.join(base_path, cond)
        rat_folders = []
        for f in os.listdir(cond_path):
            if not os.path.isdir(os.path.join(cond_path, f)):
                continue
            m = folder_re.match(f)
            if m and int(m.group(1)) == rat_id:
                rat_folders.append((f, m))

        for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
            rat_path = os.path.join(cond_path, rat_folder)
            sd_number = m.group(2)
            cond_name = m.group(3)

            pt_folders = sorted([
                f for f in os.listdir(rat_path)
                if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ])

            for pt_folder in pt_folders:
                trial_path = os.path.join(rat_path, pt_folder)
                hpc_file = find_file_local(trial_path, 'HPC_100')
                pfc_file = find_file_local(trial_path, 'PFC_100')
                state_file = find_file_local(trial_path, 'states')
                if not hpc_file or not pfc_file or not state_file:
                    continue

                try:
                    lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                    lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')
                    try:
                        ph_int, to_int, lfp_raw_r = extract_pt_intervals(lfpHPC_r, hypno_r, fs=fs)
                    except ValueError:
                        continue

                    n_ph, n_to = 0, 0
                    if ph_int is not None and len(ph_int) > 0:
                        ph_imfs, _, _ = extract_imfs_by_pt_intervals(
                            lfp_raw_r, fs, ph_int, cfg, return_imfs_freqs=True)
                        pfc_ph = extract_lfp_by_pt_intervals(lfpPFC_r, fs, ph_int)
                        phasic_imfs_all.extend(ph_imfs)
                        phasic_pfc_all.extend(pfc_ph)
                        n_ph = len(ph_imfs)

                    if to_int is not None and len(to_int) > 0:
                        to_imfs, _, _ = extract_imfs_by_pt_intervals(
                            lfp_raw_r, fs, to_int, cfg, return_imfs_freqs=True)
                        pfc_to = extract_lfp_by_pt_intervals(lfpPFC_r, fs, to_int)
                        tonic_imfs_all.extend(to_imfs)
                        tonic_pfc_all.extend(pfc_to)
                        n_to = len(to_imfs)

                    session_log.append({
                        'rat_id': rat_id, 'condition': cond, 'sd': sd_number,
                        'folder': pt_folder, 'n_phasic_intervals': n_ph,
                        'n_tonic_intervals': n_to,
                    })
                except Exception as e:
                    print(f"  [ERROR] {cond}/{pt_folder}: {e}")

    return phasic_imfs_all, phasic_pfc_all, tonic_imfs_all, tonic_pfc_all, session_log


def compute_ppc_per_rat(rat_ids, base_path=BASE_PATH, condition_filter=None,
                        fs=FS, frequencies=FREQUENCIES,
                        n_phase_bins=N_PHASE_BINS, n_cycles_wavelet=N_CYCLES_WAVELET,
                        ar_order=AR_ORDER, whiten=True,
                        min_cycles=20, save_path=None):
    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` (emd SiftConfig) before calling.")

    phase_bin_edges = np.linspace(-180, 180, n_phase_bins + 1)
    phase_centers = (phase_bin_edges[:-1] + phase_bin_edges[1:]) / 2

    kept_rats, phasic_maps, tonic_maps, phasic_ns, tonic_ns = [], [], [], [], []
    session_logs = {}

    for rat_id in rat_ids:
        print(f"\n{'='*60}\n  Rat {rat_id} — collecting intervals across all sessions\n{'='*60}")
        ph_imfs, ph_pfc, to_imfs, to_pfc, log = _collect_rat_intervals(
            rat_id, base_path, condition_filter=condition_filter, fs=fs, cfg=cfg)
        session_logs[rat_id] = log
        print(f"  phasic intervals pooled: {len(ph_imfs)}  |  tonic intervals pooled: {len(to_imfs)}")

        if len(ph_imfs) == 0 or len(to_imfs) == 0:
            print(f"  [SKIP rat {rat_id}] missing one state entirely")
            continue

        print("  -> phasic PPC")
        ppc_ph, _, n_ph = compute_field_field_ppc(
            ph_imfs, ph_pfc, fs=fs, frequencies=frequencies,
            n_phase_bins=n_phase_bins, n_cycles_wavelet=n_cycles_wavelet,
            ar_order=ar_order, whiten=whiten)
        print("  -> tonic PPC")
        ppc_to, _, n_to = compute_field_field_ppc(
            to_imfs, to_pfc, fs=fs, frequencies=frequencies,
            n_phase_bins=n_phase_bins, n_cycles_wavelet=n_cycles_wavelet,
            ar_order=ar_order, whiten=whiten)

        if ppc_ph is None or ppc_to is None or n_ph < min_cycles or n_to < min_cycles:
            print(f"  [SKIP rat {rat_id}] insufficient cycles (ph={n_ph}, to={n_to}, min={min_cycles})")
            continue

        kept_rats.append(rat_id)
        phasic_maps.append(ppc_ph)
        tonic_maps.append(ppc_to)
        phasic_ns.append(n_ph)
        tonic_ns.append(n_to)
        print(f"  [OK] rat {rat_id}: phasic={n_ph} cycles, tonic={n_to} cycles")

    results = {
        'rat_ids': np.array(kept_rats),
        'phasic_ppc': np.stack(phasic_maps) if phasic_maps else np.empty((0, len(frequencies), n_phase_bins)),
        'tonic_ppc':  np.stack(tonic_maps)  if tonic_maps  else np.empty((0, len(frequencies), n_phase_bins)),
        'phasic_n': np.array(phasic_ns),
        'tonic_n':  np.array(tonic_ns),
        'frequencies': frequencies,
        'phase_centers': phase_centers,
        'session_logs': session_logs,
        'whitened': whiten,
        'ar_order': ar_order,
    }

    print(f"\nRats kept for group analysis: {len(kept_rats)} / {len(rat_ids)}")
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"Saved to {save_path}")

    return results

## Run per-rat PPC (whitened)
Edit `rat_ids` to your full cohort. Pooling across conditions by default; set `condition_filter` to restrict.

In [ ]:
rat_ids = [3, 4, 7, 8]

results = compute_ppc_per_rat(
    rat_ids=rat_ids,
    base_path=BASE_PATH,
    condition_filter=None,
    min_cycles=20,
    whiten=True,
    ar_order=AR_ORDER,
    save_path='ppc_per_rat_results_whiten.pkl',
)

## Per-rat heatmaps (sanity check before group stats)

In [ ]:
def plot_per_rat_heatmaps(results, save_path=None):
    rats = results['rat_ids']
    freqs = results['frequencies']
    phi = results['phase_centers']
    n = len(rats)
    if n == 0:
        print('no rats kept')
        return

    fig, axes = plt.subplots(n, 3, figsize=(13, 3.2 * n), squeeze=False)
    for i, rat in enumerate(rats):
        ph = results['phasic_ppc'][i]
        to = results['tonic_ppc'][i]
        vmax = np.nanpercentile(np.stack([ph, to]), 99)
        diff = ph - to
        vmax_d = np.nanpercentile(np.abs(diff), 99)

        im0 = axes[i, 0].pcolormesh(phi, freqs, ph, shading='auto', cmap='viridis', vmin=0, vmax=vmax)
        axes[i, 0].set_title(f'Rat {rat} — phasic (n={results["phasic_n"][i]})')
        axes[i, 0].set_ylabel('Frequency (Hz)')
        plt.colorbar(im0, ax=axes[i, 0], label='PPC')

        im1 = axes[i, 1].pcolormesh(phi, freqs, to, shading='auto', cmap='viridis', vmin=0, vmax=vmax)
        axes[i, 1].set_title(f'Rat {rat} — tonic (n={results["tonic_n"][i]})')
        plt.colorbar(im1, ax=axes[i, 1], label='PPC')

        im2 = axes[i, 2].pcolormesh(phi, freqs, diff, shading='auto', cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
        axes[i, 2].set_title(f'Rat {rat} — phasic − tonic')
        plt.colorbar(im2, ax=axes[i, 2], label='ΔPPC')

    for ax in axes[-1]:
        ax.set_xlabel('Theta phase (deg)')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_per_rat_heatmaps(results)

## Paired cluster-based permutation test (phasic vs tonic, across rats)

`mne.stats.permutation_cluster_1samp_test` on the per-rat difference `phasic - tonic`, shape `(n_rats, n_freqs, n_phase_bins)`.

- **Adjacency:** lattice over (frequency × theta-phase). Frequency is a linear axis; theta-phase is **circular** so the first and last phase bins are neighbors — we build the adjacency explicitly to capture that.
- **Cluster-forming threshold:** default `None` lets MNE pick `t` for α=0.05 (two-sided) at df=n_rats-1. Override with e.g. `threshold=dict(start=0, step=0.2)` for TFCE if you prefer.
- **n_permutations:** 5000 is a reasonable default; drop to 1024 for quick iteration.
- **Tails:** `tail=0` (two-sided). Set `tail=1` if you have a directional hypothesis that phasic > tonic.

In [ ]:
from scipy import sparse

def build_freq_phase_adjacency(n_freqs, n_phase_bins, circular_phase=True):
    """4-connected lattice adjacency for the (freq, phase) grid.
    Phase axis is wrapped (circular) by default."""
    n = n_freqs * n_phase_bins
    rows, cols = [], []
    for fi in range(n_freqs):
        for pi in range(n_phase_bins):
            idx = fi * n_phase_bins + pi
            if fi + 1 < n_freqs:
                nb = (fi + 1) * n_phase_bins + pi
                rows += [idx, nb]; cols += [nb, idx]
            if pi + 1 < n_phase_bins:
                nb = fi * n_phase_bins + (pi + 1)
                rows += [idx, nb]; cols += [nb, idx]
            elif circular_phase and n_phase_bins > 2:
                nb = fi * n_phase_bins + 0
                rows += [idx, nb]; cols += [nb, idx]
    data = np.ones(len(rows), dtype=np.int8)
    return sparse.csr_matrix((data, (rows, cols)), shape=(n, n))


def paired_cluster_permutation_ppc(results, n_permutations=5000, threshold=None,
                                    tail=0, circular_phase=True, seed=0):
    X = results['phasic_ppc'] - results['tonic_ppc']  # (n_rats, n_freqs, n_phase)
    n_rats, n_freqs, n_phase = X.shape
    if n_rats < 3:
        raise ValueError(f'need >=3 rats for a group test; got {n_rats}')

    adjacency = build_freq_phase_adjacency(n_freqs, n_phase, circular_phase=circular_phase)

    t_obs, clusters, p_vals, H0 = permutation_cluster_1samp_test(
        X, n_permutations=n_permutations, threshold=threshold, tail=tail,
        adjacency=adjacency, out_type='mask', seed=seed, verbose=False,
    )
    return {
        't_obs': t_obs, 'clusters': clusters, 'p_vals': p_vals, 'H0': H0,
        'X': X, 'adjacency': adjacency,
    }

stats = paired_cluster_permutation_ppc(results, n_permutations=5000, threshold=None, tail=0)
print(f"n clusters: {len(stats['clusters'])}")
if len(stats['p_vals']):
    for i, p in enumerate(stats['p_vals']):
        print(f"  cluster {i}: p = {p:.4f}  (size={int(stats['clusters'][i].sum())} cells)")

## Group plots — mean phasic, mean tonic, mean difference with significant clusters

In [ ]:
def plot_group_ppc(results, stats, alpha=0.05, save_path=None):
    freqs = results['frequencies']
    phi = results['phase_centers']
    ph_mean = results['phasic_ppc'].mean(axis=0)
    to_mean = results['tonic_ppc'].mean(axis=0)
    diff_mean = ph_mean - to_mean

    sig_mask = np.zeros_like(diff_mean, dtype=bool)
    for cl, p in zip(stats['clusters'], stats['p_vals']):
        if p < alpha:
            sig_mask |= cl

    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    vmax = np.nanpercentile(np.stack([ph_mean, to_mean]), 99)
    vmax_d = np.nanpercentile(np.abs(diff_mean), 99)
    t_obs = stats['t_obs']
    vmax_t = np.nanpercentile(np.abs(t_obs), 99)

    im0 = axes[0].pcolormesh(phi, freqs, ph_mean, shading='auto', cmap='viridis', vmin=0, vmax=vmax)
    axes[0].set_title(f'Group mean phasic (n={len(results["rat_ids"])} rats)')
    axes[0].set_ylabel('Frequency (Hz)')
    plt.colorbar(im0, ax=axes[0], label='PPC')

    im1 = axes[1].pcolormesh(phi, freqs, to_mean, shading='auto', cmap='viridis', vmin=0, vmax=vmax)
    axes[1].set_title('Group mean tonic')
    plt.colorbar(im1, ax=axes[1], label='PPC')

    im2 = axes[2].pcolormesh(phi, freqs, diff_mean, shading='auto', cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
    axes[2].set_title('Group mean phasic − tonic')
    plt.colorbar(im2, ax=axes[2], label='ΔPPC')
    if sig_mask.any():
        axes[2].contour(phi, freqs, sig_mask.astype(float), levels=[0.5], colors='k', linewidths=1.5)

    im3 = axes[3].pcolormesh(phi, freqs, t_obs, shading='auto', cmap='RdBu_r', vmin=-vmax_t, vmax=vmax_t)
    axes[3].set_title(f't-map (paired, p<{alpha} contoured)')
    plt.colorbar(im3, ax=axes[3], label='t')
    if sig_mask.any():
        axes[3].contour(phi, freqs, sig_mask.astype(float), levels=[0.5], colors='k', linewidths=1.5)

    for ax in axes:
        ax.set_xlabel('Theta phase (deg)')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_group_ppc(results, stats, alpha=0.05)

## Phase-averaged PPC(f) with rat-level error bars

In [ ]:
def plot_phase_averaged(results, stats=None, alpha=0.05, save_path=None):
    freqs = results['frequencies']
    ph = results['phasic_ppc'].mean(axis=2)  # (n_rats, n_freqs)
    to = results['tonic_ppc'].mean(axis=2)
    n = ph.shape[0]

    ph_m, ph_se = ph.mean(0), ph.std(0, ddof=1) / np.sqrt(n)
    to_m, to_se = to.mean(0), to.std(0, ddof=1) / np.sqrt(n)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(freqs, ph_m, color='red', label=f'Phasic (n={n})')
    ax.fill_between(freqs, ph_m - ph_se, ph_m + ph_se, color='red', alpha=0.25)
    ax.plot(freqs, to_m, color='blue', label=f'Tonic (n={n})')
    ax.fill_between(freqs, to_m - to_se, to_m + to_se, color='blue', alpha=0.25)

    if stats is not None:
        sig_mask = np.zeros_like(stats['t_obs'], dtype=bool)
        for cl, p in zip(stats['clusters'], stats['p_vals']):
            if p < alpha:
                sig_mask |= cl
        freq_sig = sig_mask.any(axis=1)
        if freq_sig.any():
            y_band = ax.get_ylim()[1] * 0.98
            ax.plot(freqs[freq_sig], np.full(freq_sig.sum(), y_band), 'k|', markersize=10,
                    label=f'any-φ sig (p<{alpha})')

    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC (phase-averaged)')
    ax.set_title('HPC-PFC PPC — phasic vs tonic (per-rat mean ± SE)  — whitened')
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_phase_averaged(results, stats)

## control

In [ ]:
rat_ids = [1, 2, 6, 9]

results = compute_ppc_per_rat(
    rat_ids=rat_ids,
    base_path=BASE_PATH,
    condition_filter=None,
    min_cycles=20,
    whiten=True,
    ar_order=AR_ORDER,
    save_path='ppc_per_rat_results_whiten_control.pkl',
)

In [ ]:
plot_per_rat_heatmaps(results)

In [ ]:
stats = paired_cluster_permutation_ppc(results, n_permutations=5000, threshold=None, tail=0)
print(f"n clusters: {len(stats['clusters'])}")
if len(stats['p_vals']):
    for i, p in enumerate(stats['p_vals']):
        print(f"  cluster {i}: p = {p:.4f}  (size={int(stats['clusters'][i].sum())} cells)")

In [ ]:
plot_group_ppc(results, stats, alpha=0.05)

In [ ]:
plot_phase_averaged(results, stats)